# 🚀 NVIDIA Builder Powered Document Enrichment & Semantic Chunking Pipeline

This pipeline ingests documents from the `Data/` folder, performs high-throughput metadata extraction via **`ChatNVIDIA.abatch()`**, executes **Semantic Chunking** using **`NVIDIAEmbeddings`**, enriches each chunk with granular compliance and domain metadata via asynchronous batching (strictly throttled to **≤ 40 RPM**), and stores them in a Vector Store and structured JSON/JSONL artifacts.

In [29]:
%load_ext autoreload
%autoreload 2

import os
import json
import importlib
from dotenv import load_dotenv
import pandas as pd

# 1. Load Environment & NVIDIA API Key
load_dotenv()
print("NVIDIA API Key configured:", bool(os.environ.get("NVIDIA_API_KEY")))

NVIDIA API Key configured: True


In [30]:
import document_pipeline
importlib.reload(document_pipeline)
from document_pipeline import DocumentPipeline

# 2. Initialize the NVIDIA Document Pipeline with 40 RPM Limit
pipeline = DocumentPipeline(
    llm_model="meta/llama-3.1-8b-instruct",
    embedding_model="nvidia/nv-embedqa-e5-v5",
    temperature=0.1,
    breakpoint_threshold_type="percentile",
    batch_size=5,
    max_rpm=40  # Strictly throttled to API key quota limit
)
print("Pipeline initialized with:")
print(f"  • LLM: {pipeline.llm_model_name}")
print(f"  • Embeddings / Chunker: {pipeline.embedding_model_name}")
print(f"  • Rate Limit: {pipeline.max_rpm} RPM")

Pipeline initialized with:
  • LLM: meta/llama-3.1-8b-instruct
  • Embeddings / Chunker: nvidia/nv-embedqa-e5-v5
  • Rate Limit: 40 RPM


In [31]:
# 3. Step 1: Discover and Load Documents from Data/
raw_docs = pipeline.load_data("Data")
print(f"Loaded {len(raw_docs)} document pages/sections.")
if raw_docs:
    print("Sample document metadata:", raw_docs[0].metadata)


[Step 1] Loading documents from 'Data'...
  -> Loaded 'RBI-GUIDELINES-ON-DIGITAL-LENDING-02-09-22.pdf': 12 page(s)/section(s)
  -> Loaded 'HDFC_Bank_IR26.pdf': 628 page(s)/section(s)
Total raw pages/documents loaded: 640
Loaded 640 document pages/sections.
Sample document metadata: {'source': 'Data/Test/RBI-GUIDELINES-ON-DIGITAL-LENDING-02-09-22.pdf', 'filename': 'RBI-GUIDELINES-ON-DIGITAL-LENDING-02-09-22.pdf', 'page': 1, 'total_pages': 12, 'file_type': 'pdf'}


In [32]:
# 4. Step 2: Document-Level Metadata Extraction via .abatch()
doc_meta_map = await pipeline.extract_document_metadata_async(raw_docs)
for src, meta in doc_meta_map.items():
    print(f"\nDocument: {meta.document_title}")
    print(f"Type: {meta.document_type} | Authority: {meta.issuing_authority}")
    print(f"Domain: {meta.primary_domain}")
    print(f"Summary: {meta.executive_summary}")
    print(f"Stakeholders: {meta.key_stakeholders}")


[Step 2] Extracting document-level metadata using ChatNVIDIA.abatch()...
Found 2 unique document(s) to analyze.
  ✓ Guidelines on Digital Lending (Regulatory Guideline) - Digital Lending
  ✓ Harnessing AI in Banking (Integrated Annual Report) - Digital Lending, Fintech, Banking
Document-level metadata extraction completed in 1.56s

Document: Guidelines on Digital Lending
Type: Regulatory Guideline | Authority: Reserve Bank of India (RBI)
Domain: Digital Lending
Summary: The RBI has issued guidelines on digital lending to ensure compliance with existing regulations and to promote a smooth transition to the new guidelines. The guidelines apply to all regulated entities and require them to ensure that their lending service providers and digital lending apps comply with the guidelines. Existing digital loans must also be brought in compliance with the guidelines by November 30, 2022.
Stakeholders: ['Commercial Banks', 'Primary (Urban) Co-operative Banks', 'State Co-operative Banks', 'Dist

In [ ]:
# 5. Step 3: Semantic Chunking using NVIDIAEmbeddings
semantic_chunks = pipeline.semantic_chunking(raw_docs, doc_meta_map)
print(f"Generated {len(semantic_chunks)} semantic chunks.")
print("\n--- Sample Chunk 1 Preview ---")
print(semantic_chunks[0].page_content[:300])


[Step 3] Performing Semantic Chunking using NVIDIAEmbeddings (nvidia/nv-embedqa-e5-v5)...


In [ ]:
# 6. Step 4: Granular Metadata Enrichment via ChatNVIDIA.abatch() with 40 RPM Throttling
# Rate-limited asynchronously using a sliding 60-second window token bucket
enriched_chunks = await pipeline.enrich_chunks_async(
    chunks=semantic_chunks,
    batch_size=5,     # Dispatches 5 items per sub-batch
    max_rpm=40,       # Guarantees strictly <= 40 requests per minute
    max_retries=3     # Auto-retry with backoff on any transient network/rate limit hiccups
)
print(f"Successfully enriched {len(enriched_chunks)} chunks!")

In [ ]:
# 7. Step 5 & 6: Vector Store Indexing & Export to JSON/JSONL
vector_store = pipeline.build_vector_store(enriched_chunks)
pipeline.export_processed_data()
print("Enriched data saved to Data/processed_documents.json and Data/processed_documents.jsonl")

In [ ]:
# 8. View Processed Metadata in DataFrame
with open("Data/processed_documents.json", "r", encoding="utf-8") as f:
    data = json.load(f)

df = pd.DataFrame([
    {
        "ID": item["id"],
        "Title": item["metadata"].get("chunk_title"),
        "Summary": item["metadata"].get("chunk_summary"),
        "Entities": ", ".join(item["metadata"].get("entities", [])),
        "Compliance Mandates": len(item["metadata"].get("compliance_mandates", [])),
        "Page": item["metadata"].get("page")
    }
    for item in data
])
df.head(10)

In [ ]:
# 9. Semantic Query & Rich Metadata Retrieval Test
query = "What are the rules regarding loan disbursals and fees paid to LSPs?"
results = pipeline.search(query, k=2)

print(f"QUERY: {query}\n" + "=" * 60)
for i, doc in enumerate(results, 1):
    print(f"\n[Result {i}] {doc.metadata.get('chunk_title')}")
    print(f"Summary: {doc.metadata.get('chunk_summary')}")
    print(f"Entities: {doc.metadata.get('entities')}")
    print(f"Compliance Mandates: {doc.metadata.get('compliance_mandates')}")
    print(f"Potential Questions: {doc.metadata.get('potential_questions')}")
    print(f"Excerpt: {doc.page_content[:250]}...")